# WET-013: WEP Performance as a function of exposure time

Owner: **Bryce Kalmbach** [@jbkalmbach](https://github.com/lsst-sitcom/sitcomtn-133/issues/new?body=@jbkalmbach) <br>
Last Verified to Run: **2024-04-09** <br>
Software Version:
  - `ts_wep`: **14.1.1**
  - `lsst_distrib`: **w_2025_14**

## Test Description

This test will look at the WEP output from multiple defocal visits across a range of exposure times to investigate if increasing exposure time helps average out the atmospheric residuals.
We will calculate the average Zernikes for each visit and then find the variation in the estimates of the Zernikes from estimates on visits with the same exposure time.

In [ ]:
# Times Square Parameters
collection_name = 'u/brycek/aosBaseline_tie'
min_seq_num = 161
max_seq_num = 202
day_obs = 20241105

# Imports

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
from astropy.io import fits
from astropy import units as u
from IPython.utils import io
# from lsst.ts.wep.utils import convertZernikesToPsfWidth ### Uncomment when ts_wep available in Times Square
from astropy.table import Table, QTable, unique
from scipy.optimize import curve_fit
%matplotlib inline

In [ ]:
# Change this path to appropriate butler when on-sky images arrive
path_to_butler = '/repo/main'
butler = Butler(path_to_butler)

## Temporary functions for Times Square
These are here in the notebook until Times Square can load `ts_wep` functions.

In [ ]:
import galsim

def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM

## Load Zernike Estimates

When running exposure time tests we will run the Wavefront Estimation Pipeline (WEP) on the images. 
Once this is done all we need is the collection name used when running the pipeline and we can generate our analysis using the code below.

In [ ]:
# Load the data ids from the collection with the WEP output
data_ids = list(butler.registry.queryDataIds(('exposure', 'visit'), collections=collection_name, datasets='aggregateAOSVisitTableAvg', where=f"exposure.day_obs = {day_obs} and exposure.seq_num >= {min_seq_num} and exposure.seq_num <= {max_seq_num} and instrument = 'LSSTComCam'"))

In [ ]:
# Gather relevant visit info and zernike outputs into an Astropy table
exp_time_list = []
airmass_list = []
visit_list = []
zern_avg_list = []
z_min = 4
z_max = 28
for data_id in data_ids:
    zern_table = butler.get('aggregateAOSVisitTableAvg', dataId=data_id, collections=collection_name)
    noll_indices = zern_table.meta['nollIndices']
    zern_array = np.zeros((9, z_max - z_min + 1))
    zern_array[:, noll_indices - z_min] = zern_table['zk_CCS']
    zern_avg_list.append(np.mean(zern_array, axis=0))
    visit_list.append(data_id['visit'])
    data_id_visitInfo = data_id.to_simple().dataId
    data_id_visitInfo['detector'] = 0
    visitInfo = butler.get('postISRCCD.visitInfo', dataId=data_id_visitInfo, collections=collection_name)
    exp_time_list.append(visitInfo.exposureTime)
    airmass_list.append(visitInfo.boresightAirmass)

data_table = QTable([exp_time_list, visit_list, airmass_list, zern_avg_list], names=['exp_time', 'visit', 'airmass', 'zern_avg'])

## Exposure Time Analysis

### Examine the dataset

Just take a quick look at the various exposure times used in the data and the number of visits for each exposure time.

In [ ]:
# Comcam detector Ids
detector_ids = np.arange(9)
# Get exposure times directly from data set
exp_times = np.unique(data_table['exp_time'])

In [ ]:
data_table

In [ ]:
exp_time_counts = []
for exp_time in exp_times:
    exp_time_counts.append(np.sum(data_table['exp_time'] == exp_time))
plt.plot(exp_times, exp_time_counts, '-o')
plt.title('Number of Visits with each exposure time')
plt.ylabel('Number of Visits')
plt.xlabel('Exposure Time (seconds)')
plt.tight_layout()

### Consistency of Mean Value across Exposure Times

In this first plot we examine the mean value across the different runs. If we are in the same optical state during the different observations then we should see that the mean value will be approximately the same for each Zernike across the different exposure times. We can also separate it by detector to see if there are any effects on detectors with more vignetting than others.

In [ ]:
fig = plt.figure(figsize=(8, 5))
exp_times = np.unique(data_table['exp_time'])
for exp_time in exp_times:
    exp_time_table = data_table[data_table['exp_time'] == exp_time]
    zern_avg_array = np.array(exp_time_table['zern_avg'].value)
    plt.plot(np.arange(4, 29), convertZernikesToPsfWidth(np.mean(zern_avg_array, axis=0)), label=f'Exp Time {exp_time} sec')
    plt.xlabel('Noll Index')
    plt.ylabel('Zernike Estimate (arcsec)')
    plt.legend(fontsize=8)
plt.title('Mean Zernike Estimate on ComCam sims across Exposure Times averaged across all detectors')

### Variability in the measurements for each Zernike

Since all the means across each detector for each exposure time seem fairly consistent we can compare the variability in the measurements for each Zernike on each detector by plotting the standard deviation for each Zernike coefficient separated by the detectors.

In [ ]:
fig = plt.figure(figsize=(8,5))
for exp_time in exp_times:
    exp_time_table = data_table[data_table['exp_time'] == exp_time]
    zern_std_array = np.array(exp_time_table['zern_avg'].value)
    plt.plot(np.arange(4, 29), convertZernikesToPsfWidth(np.std(zern_std_array, axis=0)), label=f'Exp Time {exp_time} sec')
    plt.xlabel('Noll Index')
    plt.ylabel('Standard Deviation (arcsec)')
plt.legend(fontsize=8)
plt.title('Standard Deviation of the Zernike estimate across all runs at each exposure time', size=18)
plt.tight_layout()

Finally we look at the same information but take a cross section across each individual Zernike coefficient.

In [ ]:
zern_std_exp_times_all = []
for exp_time in exp_times:
    num_rows = 0
    deviations = []
    for detector in detector_ids:
        use_rows = data_table['exp_time'] == exp_time
        detector_table = data_table[use_rows]
        deviations.append(convertZernikesToPsfWidth(detector_table['zern_avg']) - np.mean(convertZernikesToPsfWidth(detector_table['zern_avg']), axis=0))
        num_rows += len(detector_table)
    deviations = np.array(deviations).reshape(num_rows, 25)
    zern_std_array = np.sqrt(1 / (len(deviations)) * np.sum(np.square(deviations), axis=0))
    zern_std_exp_times_all.append(zern_std_array)
zern_std_exp_times_all = np.array(zern_std_exp_times_all)

In [ ]:
fig = plt.figure(figsize=(20, 12))

for idx in range(25):
    fig.add_subplot(5, 5, idx+1)
    if idx+4 not in noll_indices:
        continue
    plt.scatter(exp_times, zern_std_exp_times_all[:, idx])
    plt.xlabel('Exp Time (sec)')
    plt.ylabel('Std. Dev. (arcsec)')
    plt.title(f'Z{idx+4}')
plt.suptitle('Standard Deviation as function of exposure time')
plt.tight_layout()


And then we can plot a fit and display the fit coefficients as a function of time.

In [ ]:
fig = plt.figure(figsize=(20, 12))

for idx in range(25):
    fig.add_subplot(5, 5, idx+1)
    if idx+4 not in noll_indices:
        continue
    plt.scatter(exp_times, zern_std_exp_times_all[:, idx]**2)

    def fit_func(x, a, c):
        x0 = exp_times[0]
        return c * ((x/x0)**a)

    fit_exp, fit_var = curve_fit(fit_func, exp_times, zern_std_exp_times_all[:, idx]**2)

    def fit_func_fixed_tm1(x, c):
        x0 = exp_times[0]
        return c * ((x/x0)**-1)

    fit_exp_tm1, fit_var = curve_fit(fit_func_fixed_tm1, exp_times, zern_std_exp_times_all[:, idx]**2)
    
    plt.plot(exp_times, fit_exp_tm1[0]*(exp_times / exp_times[0])**-(1), label='t^-1')
    plt.plot(exp_times, fit_exp[1]*(exp_times / exp_times[0])**fit_exp[0], label=f'Fit: t^{fit_exp[0]:.2f}')
    plt.xlabel('Exp Time (sec)')
    plt.ylabel('Variance ($arcsec^2$)')
    plt.title(f'Z{idx+4}')
    plt.legend()
plt.suptitle('Variance as function of exposure time')
plt.tight_layout()
